# Observe GEO Satellite

In [ ]:
from skyfield.api import load
from skyfield.iokit import parse_tle_file
from skyfield.api import wgs84
from datetime import timedelta

from matplotlib import pyplot as plt

In [ ]:
ts = load.timescale()

with load.open('g19.tle', 'rb') as f:
    satellites = list(parse_tle_file(f, ts))

print('Loaded', len(satellites), 'satellites')

In [ ]:
es_g19 = satellites[0]
es_g19

In [ ]:
epoch_time = es_g19.epoch.utc_jpl()
epoch_time

In [ ]:
epoch_dt = es_g19.epoch.utc_datetime()

In [ ]:
# Configuration for time increments
increment_minutes = 15
half_day = timedelta(hours=12)

# Generate list of datetimes from -0.5 days to +0.5 days from epoch_dt
start_dt = epoch_dt - half_day
end_dt = epoch_dt + half_day

dt_observables = []
observe_dt = start_dt
while observe_dt <= end_dt:
    dt_observables.append(observe_dt)
    observe_dt += timedelta(minutes=increment_minutes)

print(f"Generated {len(dt_observables)} datetime objects.")


In [ ]:
cos_ll = [+38.4, -104.82]
cos_ll_84 = wgs84.latlon(*cos_ll)

In [ ]:
loc_diff = es_g19 - cos_ll_84
topocentric = loc_diff.at(es_g19.epoch)

In [ ]:
alt, az, distance = topocentric.altaz()
print('Altitude:', alt)
print('Azimuth:', az)
print('Distance: {:.1f} km'.format(distance.km))

Let's look at the distance to the satellite over time. We note that since the altitude is a positive number, it is in view.

In [ ]:
from sat_geo_solver.observe import distance_to

distances_observed = distance_to(es_g19, cos_ll, dt_observables)

In [ ]:
plt.plot(distances_observed)
plt.ylabel("Distance to satellite (m)");

Show the rate at which the satellite is moving relative to our antenna.

In [ ]:
from sat_geo_solver.observe import range_and_rate

range_distance, rate = range_and_rate(es_g19, cos_ll, dt_observables)

In [ ]:
plt.plot(rate)
plt.ylabel("Rate of satellite (m/s)");